# Probabilities

MatchCake can efficiently compute computational-basis probabilities

$$
p(y) = \left|\langle y | V | \psi_\text{prod}\rangle\right|^2, \qquad y \in \{0,1\}^n,
$$

for any matchgate circuit $V$ applied to any qubit product state
$|\psi_\text{prod}\rangle = \bigotimes_{k=0}^{n-1}(a_k|0\rangle + b_k|1\rangle)$.
This works for computational-basis inputs as well as arbitrary product states
such as $|+\rangle^{\otimes n}$.

Both the full distribution over all $2^n$ outcomes and the marginal distribution
on a subset of wires are available through the standard `qml.probs` measurement.
Each probability reduces to a Pfaffian of the evolved Majorana covariance matrix, so
the per-outcome cost is polynomial in $n$ rather than exponential.

In [1]:
import numpy as np
import pennylane as qml

import matchcake as mc
from matchcake.operations.state_preparation import ProductState

## Matchgate circuit

We use a random single-particle transition matrix (SPTM) on 3 qubits as the
matchgate circuit $V$. The equivalent qubit unitary `U` is used only to
cross-check against `default.qubit`.

In [2]:
n = 3
wires = list(range(n))
sptm = mc.operations.SingleParticleTransitionMatrixOperation.random(wires=wires, seed=0)
U = sptm.to_qubit_operation()

## Basis state initialization

We initialize the system in a computational-basis product state using
`ProductState.from_basis_state`, compute the full distribution with `qml.probs`,
and verify against `default.qubit`.

Note that `mc.NIFDevice()` is constructed without a wire count — the device infers
its wires automatically from the queued operations.

In [3]:
state_prep = ProductState.from_basis_state([1, 0, 1], wires=wires)


@qml.qnode(mc.NIFDevice())
def circuit():
    state_prep.queue()
    sptm.queue()
    return qml.probs(wires=wires)


@qml.qnode(qml.device("default.qubit"))
def svs_circuit():
    state_prep.queue()
    U.queue()
    return qml.probs(wires=wires)


probs = np.asarray(circuit())
svs_probs = np.asarray(svs_circuit())
print(f"probs = {np.round(probs, 6)}")
print(f"sum   = {float(np.sum(probs)):.6f}")
np.testing.assert_allclose(probs, svs_probs, atol=1e-5)
print("Results agree with state-vector simulation.")

probs = [0.005412 0.       0.       0.107289 0.       0.755987 0.131312 0.      ]
sum   = 1.000000
Results agree with state-vector simulation.


## Marginal probabilities

Passing a subset of wires to `qml.probs` traces out the remaining qubits and
returns the marginal distribution on the requested wires.

In [4]:
@qml.qnode(mc.NIFDevice())
def circuit_marginal():
    state_prep.queue()
    sptm.queue()
    return qml.probs(wires=[1])


@qml.qnode(qml.device("default.qubit"))
def svs_circuit_marginal():
    state_prep.queue()
    U.queue()
    return qml.probs(wires=[1])


marginal = np.asarray(circuit_marginal())
svs_marginal = np.asarray(svs_circuit_marginal())
print(f"marginal P(q1) = {np.round(marginal, 6)}")
np.testing.assert_allclose(marginal, svs_marginal, atol=1e-5)
print("Results agree with state-vector simulation.")

marginal P(q1) = [0.7614 0.2386]
Results agree with state-vector simulation.


## Arbitrary product state initialization

`ProductState` accepts arbitrary per-qubit amplitudes $(a_k, b_k)$, enabling
superposition initial states. Here we initialize all qubits in $|+\rangle =
\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ and compute the full distribution.

In [5]:
plus_amplitudes = np.full((n, 2), 1.0 / np.sqrt(2), dtype=complex)
state_prep_plus = ProductState(plus_amplitudes, wires=wires)


@qml.qnode(mc.NIFDevice())
def circuit_plus():
    state_prep_plus.queue()
    sptm.queue()
    return qml.probs(wires=wires)


@qml.qnode(qml.device("default.qubit"))
def svs_circuit_plus():
    state_prep_plus.queue()
    U.queue()
    return qml.probs(wires=wires)


probs_plus = np.asarray(circuit_plus())
svs_probs_plus = np.asarray(svs_circuit_plus())
print(f"probs_plus = {np.round(probs_plus, 6)}")
print(f"sum        = {float(np.sum(probs_plus)):.6f}")
np.testing.assert_allclose(probs_plus, svs_probs_plus, atol=1e-5)
print("Results agree with state-vector simulation.")

probs_plus = [0.090989 0.353645 0.093942 0.024346 0.010975 0.260667 0.123998 0.041438]
sum        = 1.000000
Results agree with state-vector simulation.


## Mixed product state

Any per-qubit amplitudes are valid. Below we mix $|+\rangle$, $|0\rangle$, and
a $y$-rotated state to demonstrate the general case.

In [6]:
theta = np.pi / 3
mixed_amplitudes = np.array(
    [
        [1.0, 1.0],
        [1.0, 0.0],
        [np.cos(theta / 2), np.sin(theta / 2)],
    ],
    dtype=complex,
)
mixed_amplitudes /= np.linalg.norm(mixed_amplitudes, axis=1, keepdims=True)
state_prep_mixed = ProductState(mixed_amplitudes, wires=wires)


@qml.qnode(mc.NIFDevice())
def circuit_mixed():
    state_prep_mixed.queue()
    sptm.queue()
    return qml.probs(wires=wires)


@qml.qnode(qml.device("default.qubit"))
def svs_circuit_mixed():
    state_prep_mixed.queue()
    U.queue()
    return qml.probs(wires=wires)


probs_mixed = np.asarray(circuit_mixed())
svs_probs_mixed = np.asarray(svs_circuit_mixed())
print(f"probs_mixed = {np.round(probs_mixed, 6)}")
print(f"sum         = {float(np.sum(probs_mixed)):.6f}")
np.testing.assert_allclose(probs_mixed, svs_probs_mixed, atol=1e-5)
print("Results agree with state-vector simulation.")

probs_mixed = [0.202622 0.386695 0.077915 0.050186 0.001233 0.154149 0.093043 0.034157]
sum         = 1.000000
Results agree with state-vector simulation.


## Single-outcome probability

When only one outcome is needed, the probability of a basis state $|y\rangle$ is the
expectation value of the projector $|y\rangle\langle y|$. Measuring
`qml.expval(qml.Projector(y, wires))` evaluates a single Pfaffian directly, avoiding
the full $2^n$ enumeration.

In [7]:
target = [1, 0, 1]


@qml.qnode(mc.NIFDevice())
def circuit_projector():
    state_prep.queue()
    sptm.queue()
    return qml.expval(qml.Projector(target, wires=wires))


@qml.qnode(qml.device("default.qubit"))
def svs_circuit_projector():
    state_prep.queue()
    U.queue()
    return qml.expval(qml.Projector(target, wires=wires))


p_target = float(circuit_projector())
svs_p_target = float(svs_circuit_projector())
print(f"P({''.join(map(str, target))}) = {p_target:.6f}")
np.testing.assert_allclose(p_target, svs_p_target, atol=1e-5)
print("Results agree with state-vector simulation.")

P(101) = 0.755987
Results agree with state-vector simulation.


## Further reading

Probabilities and expectation values share the same extended Majorana / covariance
matrix machinery. For the mathematical details — the displacement vector, the
covariance matrix, the SPTM lift, and the Pfaffian formula — see
[`docs/expectation_values_theory.md`](../docs/expectation_values_theory.md).